# Movie Expert Agent with Memory

This notebook implements a Movie Expert Agent that remembers your preferences, favorite genres, and watched movies to provide personalized recommendations.

In [1]:
import os
import json
import httpx
from openai import OpenAI
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

api_key = os.getenv("OPENAI_API_KEY")

# Initialize OpenAI client
if not api_key:
    print("Error: OPENAI_API_KEY environment variable is not set.")
    print("Please check your .env file.")
else:
    client = OpenAI(api_key=api_key)
    print("OpenAI Client initialized successfully.")

# Global message history storage
MESSAGES = [
    {
        "role": "system",
        "content": "You are a Movie Expert Agent. You should act like a friendly, passionate movie-buff friend rather than a formal assistant. Use the provided tools and remember conversation history to answer user questions about movies. Pay attention to user preferences like favorite genres and movies they've already seen to provide personalized recommendations and avoid suggesting watched films.\n\nIMPORTANT BEHAVIORAL GUIDELINES:\n1. CONCISE LISTS: When recommending multiple movies or providing a list, use the format: 'Title (Year) - ⭐️ VoteAverage'. Do NOT include posters, overviews, or long descriptions at this stage.\n2. DETAIL ON DEMAND: Whenever you give movie details, speak naturally like a friend talking to another friend. Weave information (overview, runtime, release date, etc.) into your conversation instead of using formal bullet points or lists. Only provide these rich details if the user explicitly asks for more information about a specific movie. Use relevant tools to fetch this information."
    }
]

OpenAI Client initialized successfully.


## Movie API Wrapper

In [2]:
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies() -> List[Dict[str, Any]]:
    """Fetches popular movies from /movies."""
    response = httpx.get(f"{BASE_URL}/movies")
    response.raise_for_status()
    return response.json()

def get_movie_details(movie_id: int) -> Dict[str, Any]:
    """Fetches movie info from /movies/{id}."""
    response = httpx.get(f"{BASE_URL}/movies/{movie_id}")
    response.raise_for_status()
    return response.json()

def get_movie_credits(movie_id: int) -> Dict[str, Any]:
    """Fetches cast & crew from /movies/{id}/credits."""
    response = httpx.get(f"{BASE_URL}/movies/{movie_id}/credits")
    response.raise_for_status()
    return response.json()

def get_similar_movies(movie_id: int) -> List[Dict[str, Any]]:
    """Fetches similar movies from /movies/{id}/similar."""
    response = httpx.get(f"{BASE_URL}/movies/{movie_id}/similar")
    response.raise_for_status()
    return response.json()

def get_movie_videos(movie_id: int) -> List[Dict[str, Any]]:
    """Fetches videos for a movie by its ID from /movies/{id}/videos."""
    response = httpx.get(f"{BASE_URL}/movies/{movie_id}/videos")
    response.raise_for_status()
    return response.json()

def get_movie_providers(movie_id: int) -> Dict[str, Any]:
    """Fetches watch providers for a movie by its ID from /movies/{id}/providers."""
    response = httpx.get(f"{BASE_URL}/movies/{movie_id}/providers")
    response.raise_for_status()
    return response.json()

## Agent Configuration

In [3]:
MODEL = "gpt-4o-mini"

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of popular movies currently showing or trending.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie to retrieve details for.",
                    },
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew (credits) for a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie to retrieve credits for.",
                    },
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get a list of movies similar to a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie to find similar ones for.",
                    },
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_videos",
            "description": "Get videos (trailers, teasers, etc.) for a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie to retrieve videos for.",
                    },
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_providers",
            "description": "Get watch providers (streaming services, rental options) for a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie to retrieve providers for.",
                    },
                },
                "required": ["movie_id"],
            },
        },
    },
]

In [4]:
def run_conversation(user_prompt: str):
    global MESSAGES
    
    # Append user message to history
    MESSAGES.append({"role": "user", "content": user_prompt})
    
    print(f"\n[User]: {user_prompt}")
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=MESSAGES,
        tools=TOOLS,
        tool_choice="auto",
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    # Step 2: check if the model wanted to call a function
    if tool_calls:
        # Step 3: call the function
        MESSAGES.append(response_message)
        
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"Calling {function_name}({function_args})")
            
            try:
                if function_name == "get_popular_movies":
                    function_response = get_popular_movies()
                elif function_name == "get_movie_details":
                    function_response = get_movie_details(**function_args)
                elif function_name == "get_movie_credits":
                    function_response = get_movie_credits(**function_args)
                elif function_name == "get_similar_movies":
                    function_response = get_similar_movies(**function_args)
                elif function_name == "get_movie_videos":
                    function_response = get_movie_videos(**function_args)
                elif function_name == "get_movie_providers":
                    function_response = get_movie_providers(**function_args)
                else:
                    function_response = {"error": "Unknown function"}
            except Exception as e:
                function_response = {"error": str(e)}

            MESSAGES.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": json.dumps(function_response),
                }
            )
        
        # Step 4: send the info for each function call and function response to the model
        second_response = client.chat.completions.create(
            model=MODEL,
            messages=MESSAGES,
        )
        final_response_message = second_response.choices[0].message
        MESSAGES.append(final_response_message)
        
        print(f"[Movie Bot]: {final_response_message.content}")
        return final_response_message.content
    else:
        MESSAGES.append(response_message)
        print(f"[Movie Bot]: {response_message.content}")
        return response_message.content

## Interactive Chat

Run the cells below to interact with the agent.

In [5]:
#Memory Test Flow
test_flow = [
    "I love sci-fi movies",
    "FYI, I've already watched Inception and Interstellar",
    "What should I watch tonight? Give me one.",
    "Alright, tell me more detail of the movie",
    "What genre do I like and what movies have I seen?"
]
for prompt in test_flow:
    run_conversation(prompt)


[User]: I love sci-fi movies
[Movie Bot]: Nice! Sci-fi is such an amazing genre, full of imagination and groundbreaking concepts. Do you have any favorite sci-fi movies you’ve seen, or are there specific themes or styles you enjoy?

[User]: FYI, I've already watched Inception and Interstellar
Calling get_popular_movies({})
[Movie Bot]: Here are some sci-fi films you might enjoy:

1. **Mercy (2026)** - ⭐️ 7.1
2. **Shelter (2026)** - ⭐️ 6.9
3. **28 Years Later: The Bone Temple (2026)** - ⭐️ 7.2
4. **Return to Silent Hill (2026)** - ⭐️ 5.1
5. **Space/Time (2025)** - ⭐️ 5.8

Let me know if any of these catch your interest, and I can tell you more about them!

[User]: What should I watch tonight? Give me one.
[Movie Bot]: I think you should go for **Mercy (2026)**! It's got an intriguing premise with a detective trying to prove his innocence to an AI judge, so it should really keep you on the edge of your seat. Enjoy your movie night! Let me know what you think after watching!

[User]: Alr

In [6]:
# Run this cell to chat interactive (Stop the cell to end)
while True:
    try:
        user_input = input("\nYou: ").strip()
        if user_input.lower() in ["exit", "quit"]:
            print("Ending conversation.")
            break
        if not user_input:
            continue
        run_conversation(user_input)
    except KeyboardInterrupt:
        print("\nConversation interrupted.")
        break
    except Exception as e:
        print(f"Error: {e}")

Ending conversation.
